In [1]:
import pandas as pd
import json
import ollama

print("Biblioteker lastet inn!")


Biblioteker lastet inn!


In [2]:
# De 5 kategoriene vi skal sortere diagnosene i
KATEGORIER = """
1. Hysteria - klassisk hysteria-diagnose
2. Hysterisk personlighet - personlighetstrekk, ikke akutt tilstand
3. Ex constitutione - konstitusjonell/medfødt svakhet
4. Epilepsi-relatert - epilepsi med hysteriske trekk eller omvendt, inkludert kramper
5. Annet - passer ikke i noen av kategoriene over
"""

# Last inn rensede data
with open("../data/diagnoses_clean.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(f"Lastet {len(df)} diagnoser")
print(f"\nKategorier:\n{KATEGORIER}")

Lastet 3103 diagnoser

Kategorier:

1. Hysteria - klassisk hysteria-diagnose
2. Hysterisk personlighet - personlighetstrekk, ikke akutt tilstand
3. Ex constitutione - konstitusjonell/medfødt svakhet
4. Epilepsi-relatert - epilepsi med hysteriske trekk eller omvendt, inkludert kramper
5. Annet - passer ikke i noen av kategoriene over



In [4]:
def klassifiser_diagnose(diagnose_tekst):
    prompt = f"""Du er en medisinsk historiker som kategoriserer historiske diagnosestrenger.

Kategorier:
{KATEGORIER}

Diagnose: "{diagnose_tekst}"

Svar kun med nummeret (1-5) og kategorinavnet. Eksempel: "3. Ex constitutione"
Ikke skriv noe annet."""

    respons = ollama.chat(
        model="gemma4:12b",
        messages=[{"role": "user", "content": prompt}]
    )
    
    return respons["message"]["content"].strip()

# Test med én diagnose
test = klassifiser_diagnose("Hysteria med kramper")
print(f"Testresultat: {test}")

Testresultat: 4. Epilepsi-relatert


In [6]:
#Plukke opp der den stoppa

import os

TEMP_FILE = "../data/klassifisert_temp.json"

if os.path.exists(TEMP_FILE):
    df_ferdig = pd.read_json(TEMP_FILE)
    ferdig_ids = set(df_ferdig["id"].tolist())
    df_gjenstår = df[~df["id"].isin(ferdig_ids)].copy().reset_index(drop=True)
    print(f"Fortsetter fra temp-fil: {len(df_ferdig)} ferdig, {len(df_gjenstår)} gjenstår")
else:
    df_ferdig = pd.DataFrame()
    df_gjenstår = df.copy().reset_index(drop=True)
    print(f"Starter fra scratch: {len(df_gjenstår)} poster")

Fortsetter fra temp-fil: 2250 ferdig, 853 gjenstår


In [7]:
#Husk, bytt funksjon i løkka: alle eller gjentående: 
#Og bytt df.iterrows() til df_gjenstår.iterrows() i løkka.

print("Kategoriserer alle diagnoser...")

resultater = []
for i, (_, rad) in enumerate(df_gjenstår.iterrows()):
    kategori = klassifiser_diagnose(rad["diagnosis_clean"])
    resultater.append(kategori)
    print(f"  [{i+1}/{len(df_gjenstår)}] {rad['id']} → {kategori}")
    
    # Lagre for hver 50. post
    if (i + 1) % 50 == 0:
        df_temp = df_gjenstår.iloc[:i+1].copy()
        df_temp["kategori"] = resultater
        df_combined = pd.concat([df_ferdig, df_temp], ignore_index=True)
        df_combined.to_json("../data/klassifisert_temp.json",
                        orient="records", force_ascii=False, indent=2)
        print(f"  💾 Lagret fremgang ({len(df_combined)} poster totalt)")

# Lagre endelig resultat
df_gjenstår["kategori"] = resultater
df_final = pd.concat([df_ferdig, df_gjenstår], ignore_index=True)
df_final.to_json("../data/klassifisert.json", orient="records", force_ascii=False, indent=2)
print(f"\nFerdig! Totalt {len(df_final)} diagnoser klassifisert.")

Kategoriserer alle diagnoser...
  [1/853] 9a34ab5f-2e0d-418e-a1fa-20d9d8f09cc1 → 2. Hysterisk personlighet
  [2/853] bef426bf-ef36-4f3b-aa62-edee60cdc5c4 → 1. Hysteria
  [3/853] c6d549cd-4efc-4a1b-8713-cd128fc4601f → 2. Hysterisk personlighet
  [4/853] f1135873-bda8-4441-a7b6-2a3492f6bd4f → 2. Hysterisk personlighet
  [5/853] 8e467d0d-f562-4da1-b7e1-a72c3c5b6685 → 1. Hysteria
  [6/853] de53df77-b22d-4139-8f08-3df836dba30f → 2. Hysterisk personlighet
  [7/853] c891d33e-06bd-44e7-8a08-dff8fc4fb7b4 → 5. Annet
  [8/853] cd3b7ea2-13b1-4d52-83ec-b9183ca37df7 → 2. Hysterisk personlighet
  [9/853] e14ec374-499f-4891-ba6e-74e8590f62d3 → 2. Hysterisk personlighet
  [10/853] d9543797-f48d-481b-a7c7-1b4444993aa4 → 2. Hysterisk personlighet
  [11/853] dacc90a4-85be-474b-912b-8629440a1550 → 2. Hysterisk personlighet
  [12/853] e8c18da2-6c87-4e23-aa9f-f982b7fae12e → 1. Hysteria
  [13/853] eb584945-3ec3-4195-95c0-838316eeeeae → 2. Hysterisk personlighet
  [14/853] e31957f5-da4d-4095-8ecf-cea22ee7857f 

In [7]:
# Slå sammen variantene av kategori 1
# Merk: Gemma returnerte av og til ekstra tekst i kategori 1.
# Dette normaliseres her for konsistent kategorisering.

df_ferdig["kategori"] = df_ferdig["kategori"].str.replace(
    r"^1\. Hysteria.*", "1. Hysteria", regex=True
)
df_ferdig.to_json("../data/klassifisert.json", orient="records", force_ascii=False, indent=2)
print("Fikset! Ny fordeling:")
print(df_ferdig["kategori"].value_counts())

Fikset! Ny fordeling:
kategori
1. Hysteria                  1583
3. Ex constitutione          1133
2. Hysterisk personlighet     203
5. Annet                      134
4. Epilepsi-relatert           50
Name: count, dtype: int64


In [8]:
#Sjekk at alle data er lagret

import pandas as pd

df_ferdig = pd.read_json("../data/klassifisert.json")
print(f"Antall klassifiserte: {len(df_ferdig)}")
print(f"\nKolonner: {df_ferdig.columns.tolist()}")
print(f"\nFordeling per kategori:")
print(df_ferdig["kategori"].value_counts())
df_ferdig.head()

Antall klassifiserte: 3103

Kolonner: ['id', 'diagnosis', 'diagnosis_clean', 'kategori']

Fordeling per kategori:
kategori
1. Hysteria                  1583
3. Ex constitutione          1133
2. Hysterisk personlighet     203
5. Annet                      134
4. Epilepsi-relatert           50
Name: count, dtype: int64


,id,diagnosis,diagnosis_clean,kategori
0,04DD8FF1-66C0-453E-BFCF-A576624862E6,hysteria ?,hysteria ?,1. Hysteria
1,10478F81-98D2-4F83-8EA6-1EDF1811680C,hysteria 7,hysteria 7,1. Hysteria
2,97883cca-48aa-46f1-bab0-57f234b2f598,hysteria,hysteria,1. Hysteria
3,E8F128A2-502F-47E8-B73F-6566E52D4B95,202|hysteria,202|hysteria,1. Hysteria
4,8F1CC767-D817-4334-B717-D493D2EBC80B,hysteria,hysteria,1. Hysteria


NameError: name 'plt' is not defined